In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy
from scipy.stats import ttest_ind
import seaborn as sns
from math import sqrt
import re
from matplotlib.ticker import ScalarFormatter
import matplotlib.ticker as mticker
from matplotlib.colors import to_rgb, to_hex

cell_types = ["Exc", "Pvalb", "Sst", "Vip", "Htr3a"]
colors = [
    "tab:red",
    "tab:blue",
    "tab:green",
    "tab:purple",
    "tab:purple",
]
layer_colors = [
    "tab:red",
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:purple",
]
layers = ["L1", "L2/3", "L4", "L5", "L6"]

from collections import OrderedDict

Perturb_dict = OrderedDict()
Perturb_dict["no perturb"] = ""

In [ ]:
net_name = "core"  # change to "core_<variant>" for a model variant
mua_power_all = pd.read_csv(
    f"{net_name}/metrics/csv_files_other_visual_stim_memory_efficient/other_visual_stim_all_dirs_mua_power.csv"
)

In [ ]:
def str2list(str_time):
    str_time = str_time[1:-1]
    array_list = str_time.strip(",").split(", ")
    return np.stack([float(ii) for ii in array_list])

mua_power_all["f"] = mua_power_all["f"].apply(str2list)
mua_power_all["power"] = mua_power_all["power"].apply(str2list)
mua_power_all["timestamps"] = mua_power_all["timestamps"].apply(str2list)

In [ ]:
mua_power_all.dir.value_counts()

In [ ]:
# get power_df, where the results are averaged across 10 trials
power_df = mua_power_all.groupby(["condition", "layer", "cell"]).agg(
    {
        "power": list,
        "f": list,
    }
)
power_df.reset_index(inplace=True)
power_df["mean"] = power_df["power"].apply(np.mean, axis=0)
power_df["std"] = power_df["power"].apply(np.std, axis=0)
power_df["f"] = power_df["f"].apply(np.mean, axis=0)

In [ ]:
def ps_for_MUA_df(row, bins=np.linspace(500, 3000, 501), nperseg=100):
    spikes = row["timestamps"][0]
    (counts, bins) = np.histogram(spikes, bins=bins)
    binsize = bins[1] - bins[0]
    mua = counts / (binsize / 1e3) # (n_neuron) # Hz - normalized by thr total number of neurons
    f, s = scipy.signal.welch(
        mua, fs=1 / binsize * 1e3, nperseg=nperseg, scaling="spectrum"
    )
    return f, s, mua

In [ ]:
mua = (
    mua_power_all.groupby(["condition", "trial", "layer", "dir"])["timestamps"]
    .agg(list)
    .to_frame()
)
mua.reset_index(inplace=True)
bins = np.linspace(500, 3000, 1250)  # binsize: (3000-500)/500 = 5ms
nperseg = 500  # segment length: 200*5ms = 1s 
mua[["f", "power", "mua"]] = mua.apply(
    ps_for_MUA_df,
    bins=bins,
    nperseg=nperseg,
    axis=1,
    result_type="expand",
)

In [ ]:
mua.dir.value_counts()

In [ ]:
laminar_power_df = mua.groupby(["condition", "layer"]).agg(
    {
        "power": list,
        "f": list,
    }
)
laminar_power_df.reset_index(inplace=True)
laminar_power_df["mean"] = laminar_power_df["power"].apply(np.mean, axis=0)
laminar_power_df["std"] = laminar_power_df["power"].apply(np.std, axis=0)
laminar_power_df["f"] = laminar_power_df["f"].apply(np.mean, axis=0)

In [ ]:
# layer_mua_power_all = pd.read_csv("core/metrics/csv_files_other_visual_stim_memory_efficient/other_visual_stim_all_dirs_mua_power_layer.csv")

# layer_mua_power_all["power"] = layer_mua_power_all["power"].apply(str2list)
# layer_mua_power_all["f"] = layer_mua_power_all["f"].apply(str2list)

In [ ]:
# layer_mua_power_all.dir.value_counts()

In [ ]:
# laminar_power_df = layer_mua_power_all.groupby(["layer", "condition"]).agg(
#     {
#         "power": list,
#         "f": list,
#     }
# )

In [ ]:
laminar_power_df.power.apply(np.size,axis=0)

In [ ]:
# # get power_df, where the results are averaged across 10 trials
# laminar_power_df = layer_mua_power_all.groupby(["layer", "condition"]).agg(
#     {
#         "power": list,
#         "f": list,
#     }
# )
# laminar_power_df.reset_index(inplace=True)
# laminar_power_df["mean"] = laminar_power_df["power"].apply(np.mean, axis=0)
# laminar_power_df["std"] = laminar_power_df["power"].apply(np.std, axis=0)
# laminar_power_df["f"] = laminar_power_df["f"].apply(np.mean, axis=0)
# laminar_power_df.drop(columns=["power"], inplace=True)

# mean_power_cond = laminar_power_df.groupby(["condition"]).agg(
#     {
#         "mean": list,
#     }
# )
# mean_power_cond["max"] = mean_power_cond["mean"].apply(
#     lambda row: np.max(np.vstack(np.asanyarray(row)), axis=0)
# )
# mean_power_cond.rename(columns={"index": "condition"})
# mean_power_cond.reset_index(inplace=True)


# def get_norm_power(row):
#     max_values = mean_power_cond.loc[
#         mean_power_cond["condition"].isin([row["condition"]]).values,
#         "max",
#     ].values[0]
#     # print(row["contrast"])
#     norm_power = row["mean"] / max_values
#     # print(np.max(norm_power))
#     return norm_power


# laminar_power_df["norm power"] = laminar_power_df.apply(
#     lambda row: get_norm_power(row), axis=1
# )

In [ ]:
mua_power_all.condition.unique().tolist()

In [ ]:
colors

In [ ]:

selected_cells = ["Exc", "Pvalb", "Sst", "Vip"]

fig, axes = plt.subplots(
    ncols=5, nrows=5, sharex=True, figsize=(9.5, 6), dpi=300
)  # , sharey="col"
fig.patch.set_facecolor("none")
f = power_df.loc[0, "f"]

indx = (f >= 4) & (f <= 100)

colors = np.delete(plt.colormaps["plasma"](np.linspace(0, 1, 8)), 7, axis=0)
# np.delete(plt.colormaps["Accent_r"](np.linspace(0, 1, 8)), 4, axis=0)

for (jj, cell), c in zip(enumerate(selected_cells), colors):
    data = power_df.query(
        'cell == "' + cell + '"'
    )
    print("cell {} layer_count {}".format(cell, data.layer.value_counts()))
    data.reset_index(inplace=True)
    # Hide first row axis
    axes[0, jj].get_xaxis().set_visible(False)
    axes[0, jj].get_yaxis().set_visible(False)
    axes[0, jj].set_axis_off()

    for ii, layer in enumerate(layers[1:]):
        ax = axes[ii + 1, jj] # 'natural movie one',  'natural movie two',  
        for ci, condition in enumerate(['natural scenes', 'natural movie one',  'natural movie two', 'drifting grating']):
            m = data.loc[
                (data["layer"] == layer).values & (data["condition"] == condition).values,
                "mean",
            ].values
            s = data.loc[
                (data["layer"] == layer).values & (data["condition"] == condition).values,
                "std",
            ].values / sqrt(
                10
            )  # standard error
            m, s = m[0], s[0]
            ax.plot(
                f[indx],
                m[indx],
                color=colors[ci],
                linewidth=1,
                label="{}".format(condition),
            )
            ax.fill_between(
                f[indx],
                m[indx] - 1.96 * s[indx],
                m[indx] + 1.96 * s[indx],
                facecolor=colors[ci],
                alpha=0.4,
            )

            sns.despine()  # Removes top and right spines by default
            ax.spines["left"].set_position(("outward", 3))
            ax.spines["bottom"].set_position(("outward", 3))

            ax.set_xlim([4, 100])
            ax.set_xticks([4, 20, 40, 60, 80, 100])
            ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
            ax.set_xticks([10, 30, 50, 70, 90], minor=True)
            ax.tick_params(axis='x', which='major', length=5)
            ax.tick_params(axis='x', which='minor', length=3)

            # Format y-axis with scientific notation for large values
            formatter = ScalarFormatter(useMathText=True)
            formatter.set_powerlimits((-3, 3))
            ax.yaxis.set_major_formatter(formatter)
            ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
            ax.tick_params(axis='y', labelsize=8)

            ax.text(
                1,
                0.95,
                cell + " " + layer,
                horizontalalignment="right",
                verticalalignment="top",
                fontsize=10,
                alpha=1,
                transform=ax.transAxes,
                fontweight="bold",
            )

        if layer == "L2/3":
            # Place legend in the row above (index ii-1, since L2/3 is at ii=0 for layers[1:])
            legend_ax = axes[0, jj]  # First row for this column
            legend_ax.legend(
                handles=ax.get_legend_handles_labels()[0],
                labels=ax.get_legend_handles_labels()[1],
                loc="center",
                bbox_to_anchor=(0.5, 0.5),
                mode=None,
                borderaxespad=0.0,
                fontsize=6,
            )

data = laminar_power_df

jj += 1
# Hide first row axis
axes[0, jj].get_xaxis().set_visible(False)
axes[0, jj].get_yaxis().set_visible(False)
axes[0, jj].set_axis_off()
for ii, layer in enumerate(layers[1:]):
    ax = axes[ii + 1, jj] # 'natural movie one',  'natural movie two',  
    for ci, condition in enumerate(['natural scenes', 'natural movie one',  'natural movie two', 'drifting grating']):
        m = data.loc[
            (data["layer"] == layer).values & (data["condition"] == condition).values,
            "mean",
        ].values
        s = data.loc[
            (data["layer"] == layer).values & (data["condition"] == condition).values,
            "std",
        ].values / sqrt(
            10
        )  # standard error
        m, s = m[0], s[0]
        ax.plot(
            f[indx],
            m[indx],
            color=colors[ci],
            linewidth=1,
            label="{}".format(condition),
        )
        ax.fill_between(
            f[indx], m[indx] - 1.96 * s[indx], m[indx] + 1.96 * s[indx], facecolor=colors[ci], alpha=0.4
        )

        sns.despine()  # Removes top and right spines by default
        ax.spines["left"].set_position(("outward", 3))
        ax.spines["bottom"].set_position(("outward", 3))

        ax.set_xlim([4, 100])
        ax.set_xticks([4, 20, 40, 60, 80, 100])
        ax.set_xticklabels([4, 20, 40, 60, 80, 100], fontsize=8)
        ax.set_xticks([10, 30, 50, 70, 90], minor=True)
        ax.tick_params(axis='x', which='major', length=5)
        ax.tick_params(axis='x', which='minor', length=3)

        # Format y-axis with scientific notation for large values
        formatter = ScalarFormatter(useMathText=True)
        formatter.set_powerlimits((-3, 3))
        ax.yaxis.set_major_formatter(formatter)
        ax.ticklabel_format(axis="y", style="scientific", scilimits=(-3, 3))
        ax.tick_params(axis='y', labelsize=8)

        ax.text(
            1,
            0.95,
            layer,
            horizontalalignment="right",
            verticalalignment="top",
            fontsize=10,
            alpha=1,
            transform=ax.transAxes,
            fontweight="bold",
        )

    if layer == "L2/3":
        # Place legend in the row above (index ii-1, since L2/3 is at ii=0 for layers[1:])
        legend_ax = axes[0, jj]  # First row for this column
        legend_ax.legend(
            handles=ax.get_legend_handles_labels()[0],
            labels=ax.get_legend_handles_labels()[1],
            loc="center",
            bbox_to_anchor=(0.5, 0.5),
            mode=None,
            borderaxespad=0.0,
            fontsize=6,
        )

fig.supxlabel("Frequency [Hz]")
fig.supylabel("Power [(spks/s)²]")
fig.tight_layout()
fig.savefig(
    "paper_figs/other_visual_stim_power_Jul27_2026.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
